# Đánh giá chương trình khuyến mãi: +39% doanh số có đáng tiền không?

Bộ `store-sales` (Rossmann): `train.csv` (doanh số, lượt khách, cờ `Promo` theo ngày của 1 115 cửa hàng,
01/01/2013 tới 31/07/2015) và `store.csv` (`StoreType`, `Assortment`, `Promo2`...). `Promo = 1` là ngày cửa hàng chạy
khuyến mãi; lịch do công ty quyết định, chạy đồng loạt toàn chuỗi, thứ Hai tới thứ Sáu, thường cách tuần.

**Bối cảnh.** Commercial Director chuẩn bị ngân sách năm sau. Dashboard ghi: "ngày khuyến mãi bán nhiều hơn 39%".
Bà hỏi: **"Khuyến mãi có đáng tiền không? Nên giữ, mở rộng, hay cắt ở loại cửa hàng / ngày nào? Giảm giá sâu tới
đâu thì vẫn có lãi?"** Dữ liệu không có giá, mức giảm giá hay giá vốn; bạn phải nói rõ phần nào là đo được, phần nào
là giả định, và phần nào cần một thử nghiệm mới trả lời được.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` dùng cho phân tích tự do; khi chấm chúng bị bỏ qua.
- Viết `MEMO.md` (cùng thư mục với notebook) theo khung có sẵn.
- Chấm phần tự động: `uv run pytest tracks/data-analyst/modules/13-business-financial-analytics/exercises/advanced/02-promo-review -q`. Phần memo chấm theo rubric trong lời giải.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc và ghép thông tin cửa hàng
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

DATA = ensure("store-sales")
train = pd.read_csv(DATA / "train.csv", parse_dates=["Date"], dtype={"StateHoliday": str})
stores = pd.read_csv(DATA / "store.csv")
train = train.merge(stores[["Store", "StoreType", "Assortment", "Promo2"]], on="Store", how="left")
print(train.shape)
print(pd.crosstab(train.DayOfWeek, train.Promo))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Lift của khuyến mãi, so sánh công bằng

`promo_lift(train, by=None) -> pd.DataFrame`: chỉ dùng các ngày-cửa hàng **mở cửa có bán** (`Open == 1` và
`Sales > 0`) và **thứ Hai tới thứ Sáu** (`DayOfWeek` 1..5, vì khuyến mãi không bao giờ chạy thứ Bảy, Chủ nhật; so
mọi ngày là so ngày thường có khuyến mãi với cả cuối tuần không có). Với mỗi nhóm của cột `by` (hoặc một nhóm duy
nhất tên `"all"` khi `by=None`), so trung bình mỗi ngày-cửa hàng có khuyến mãi với không có:

| Cột | Định nghĩa |
| --- | --- |
| `sales_lift` | `mean(Sales | Promo=1) / mean(Sales | Promo=0) − 1` |
| `customers_lift` | như trên với `Customers` |
| `spc_lift` | `(1 + sales_lift) / (1 + customers_lift) − 1` (chi tiêu mỗi khách) |
| `promo_days`, `base_days` | số ngày-cửa hàng có / không có khuyến mãi |

Index tên `by` (hoặc không tên khi `by=None`), tăng dần.

`breakeven_discount(lift, margin) -> float`: mức giảm giá tối đa d (tiền giảm giá chia cho doanh số ngày khuyến
mãi) để lợi nhuận gộp ngày khuyến mãi không thấp hơn ngày thường, khi doanh số tăng `lift` và gross margin trước
giảm giá là `margin`. Hàng bán ra có giá niêm yết S1 + D với D = d · S1, giá vốn (1 − margin)(S1 + D), nên

$$
d^{*} = \frac{m \, L}{(1 + L)(1 - m)}
$$

In [ ]:
def promo_lift(train: pd.DataFrame, by: str | None = None) -> pd.DataFrame:
    """Lift doanh số, lượt khách, chi tiêu mỗi khách của ngày khuyến mãi (thứ Hai - thứ Sáu, mở cửa có bán)."""
    # TODO
    raise NotImplementedError


def breakeven_discount(lift: float, margin: float) -> float:
    """Mức giảm giá (theo doanh số ngày khuyến mãi) để lợi nhuận gộp không đổi."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: con số trên dashboard và con số so sánh công bằng
open_days = train[(train.Open == 1) & (train.Sales > 0)]
naive = open_days.groupby("Promo").Sales.mean()
print(f"dashboard (mọi ngày mở cửa): {naive[1] / naive[0] - 1:+.1%}")

In [ ]:
# explore: TODO - lift theo loại cửa hàng, danh mục, thứ trong tuần

In [ ]:
# explore: khuyến mãi có kéo cầu từ ngày khác không? Cuối tuần của tuần có / không có khuyến mãi
d = open_days.assign(week=open_days.Date - pd.to_timedelta(open_days.Date.dt.day_of_week, unit="D"))
promo_week = d.groupby("week").Promo.max()
d["promo_week"] = d.week.map(promo_week)
weekend = d[d.DayOfWeek >= 6].groupby(["DayOfWeek", "promo_week"]).Sales.mean().unstack()
print((weekend[1] / weekend[0] - 1).round(4).rename("cuối tuần của tuần khuyến mãi so với tuần thường"))
# tuần thường ngay sau một tuần khuyến mãi so với tuần thường sau một tuần thường (hiếm: quanh lễ)
prev = promo_week.shift(1)
wd = d[(d.DayOfWeek <= 5) & (d.promo_week == 0)].assign(prev=lambda x: x.week.map(prev))
print(wd.groupby("prev").agg(sales=("Sales", "mean"), weeks=("week", "nunique")))

In [ ]:
# explore: theo tuần - tuần khuyến mãi so với tuần thường (cửa hàng đủ ngày, tuần đủ 7 ngày)
full = train.groupby("Store").Date.nunique()
panel = train[train.Store.isin(full[full == full.max()].index)]
wk = panel.assign(week=panel.Date - pd.to_timedelta(panel.Date.dt.day_of_week, unit="D"))
wk = wk.groupby("week").agg(sales=("Sales", "sum"), promo=("Promo", "max"), days=("Date", "nunique"))
wk = wk[wk.days == 7]
print(wk.groupby("promo").sales.mean().pipe(lambda s: s[1] / s[0] - 1))

In [ ]:
# explore: Promo2 (coupon dài hạn) - cửa hàng tham gia bán ít hơn: do Promo2 hay do chọn cửa hàng?
print(open_days.groupby("Promo2").Sales.mean().round(0))
s2 = stores.dropna(subset=["Promo2SinceYear"])
print(s2.Promo2SinceYear.value_counts().sort_index())

In [ ]:
# explore: TODO - mức giảm giá hoà vốn theo vài giả định gross margin, theo loại cửa hàng

In [ ]:
# explore: TODO - 2-3 chart cho memo, tiêu đề là kết luận có con số

## 2. Viết memo

Mở `MEMO.md`, viết theo khung (200-600 từ, dòng đầu là headline có con số). Người đọc là Commercial Director: bà
cần biết khuyến mãi tác động bao nhiêu (đo thế nào), ở đâu mạnh / yếu, điều kiện để có lãi, và bước tiếp theo để
trả lời phần dữ liệu hiện có không trả lời được.